# Model Context Protocol (MCP)

**Domain:** Agentic AI  ·  **runnable:** yes

A refresher on MCP — the open protocol that standardizes how LLM applications connect to external tools, data, and prompts.


## 1. What & Why

**Model Context Protocol (MCP)** is an open standard (introduced by Anthropic in late 2024) for connecting LLM applications to external capabilities. It defines a wire protocol — JSON-RPC 2.0 over a transport — so that any compliant **host/client** can talk to any compliant **server** without bespoke glue code.

**The problem it solves: the M×N integration explosion.** Before MCP, every agent framework had its own way to define tools, and every data source (GitHub, Postgres, Slack, your filesystem) had to be re-wrapped for each framework. With *M* applications and *N* integrations you wrote *M×N* adapters. MCP turns this into *M+N*: a tool author writes **one** MCP server, and it works in Claude Desktop, Cursor, Zed, your custom agent, etc. Think of it as **"USB-C for AI applications"** — a universal port between models and the world.

**Reach for MCP when:**
- You want a capability (a tool, a data source, a prompt template) to be reusable across multiple agents/clients.
- You're building a host (IDE, chat app, agent runtime) and want to plug in a growing ecosystem of integrations.
- You need a clean trust boundary: the server runs as a separate process, so it can have its own permissions and credentials.

**Skip it when:**
- You have a single app with a handful of in-process functions — a plain function-calling/tool-use loop is simpler. MCP adds a process boundary and protocol overhead you don't need yet.
- The "tool" is pure local computation tightly coupled to your app's internals.


## 2. Mental Model

MCP is **client–server, like the Language Server Protocol (LSP) but for AI tools.** LSP let one editor extension (say, a Python language server) work in VS Code, Vim, and Emacs alike. MCP does the same for model context.

```
        ┌─────────────────────── Host (e.g. Claude Desktop, Cursor, your agent) ──────────────────────┐
        │   The host embeds the LLM and spawns one MCP *client* per server it connects to.             │
        │                                                                                              │
        │   ┌──────────┐   JSON-RPC 2.0   ┌────────────────┐                                            │
        │   │  Client  │ <==============> │  MCP Server A   │  e.g. filesystem  (exposes tools/resources)│
        │   └──────────┘   over stdio/HTTP└────────────────┘                                            │
        │   ┌──────────┐                  ┌────────────────┐                                            │
        │   │  Client  │ <==============> │  MCP Server B   │  e.g. GitHub                                │
        │   └──────────┘                  └────────────────┘                                            │
        └──────────────────────────────────────────────────────────────────────────────────────────────┘
```

- **Host**: the application the user interacts with; it owns the LLM and decides what to expose to it.
- **Client**: a connector inside the host — exactly one per connected server, maintaining a 1:1 session.
- **Server**: a separate program exposing capabilities. It does *not* call the LLM; it just answers protocol requests.

Key inversion to remember: **the server provides capabilities, the host/model decides whether and when to use them.** The server is "dumb and honest"; intelligence lives in the host.


## 3. Key Concepts

The protocol exposes three **server primitives** plus a couple of client-side ones:

| Primitive | Controlled by | What it is |
|---|---|---|
| **Tools** | *model* | Functions the LLM can call (with side effects) — `search_issues`, `run_query`. Each has a name, description, and JSON-Schema `inputSchema`. This is the workhorse. |
| **Resources** | *application* | Read-only data the host can load into context, addressed by URI (`file:///…`, `postgres://…/schema`). Like GET endpoints — no side effects. |
| **Prompts** | *user* | Reusable, parameterized prompt templates the user invokes (e.g. a slash-command "summarize this PR"). |

Client-side primitives (server can ask the host for these):
- **Sampling** — the server requests an LLM completion *through the host*, so the server doesn't need its own API key.
- **Roots** — the host tells the server which filesystem/URI scopes it's allowed to touch.
- **Elicitation** — the server asks the host to prompt the user for missing input.

Other essentials:
- **JSON-RPC 2.0** — every message is a request/response/notification with `jsonrpc`, `id`, `method`, `params`.
- **Lifecycle** — connection opens with an `initialize` handshake that exchanges **capabilities** and protocol version, then `tools/list`, `tools/call`, etc.
- **Transports** — **stdio** (server is a child process; the default for local servers) and **Streamable HTTP** (for remote servers; supersedes the older HTTP+SSE transport).
- **Capability negotiation** — both sides advertise what they support during `initialize`, so neither assumes unsupported features.


## 4. Setup

MCP itself is language-agnostic (it's just JSON-RPC). The official Python SDK is `mcp`, which bundles **FastMCP** for ergonomically authoring servers.

```bash
pip install "mcp[cli]"        # Python SDK + the `mcp` dev CLI
# or, the TypeScript SDK:  npm install @modelcontextprotocol/sdk
```

The worked examples below are **pure-Python simulations of the wire protocol** — they need no installs and run anywhere, so you can see exactly what flows over the wire. The final cell shows the *real* SDK shape and runs only if `mcp` is installed.


In [ ]:
# Install is optional — the core examples are dependency-free.
# Uncomment to get the real SDK used in the last cell:
# %pip install "mcp[cli]"

import importlib.util
print("mcp SDK installed:", importlib.util.find_spec("mcp") is not None)


## 5. Worked Examples

### Example 1 — A minimal MCP server, by hand

To demystify the protocol, here's a tiny "server" that handles the three messages every session uses: `initialize`, `tools/list`, and `tools/call`. Real servers add resources, prompts, error handling, and a transport — but the message shapes are exactly these.


In [ ]:
import json

# --- A toy MCP server: a tool registry + a JSON-RPC dispatcher ---------------
TOOLS = {
    "add": {
        "description": "Add two numbers.",
        "inputSchema": {
            "type": "object",
            "properties": {"a": {"type": "number"}, "b": {"type": "number"}},
            "required": ["a", "b"],
        },
        "fn": lambda a, b: a + b,
    },
    "reverse": {
        "description": "Reverse a string.",
        "inputSchema": {
            "type": "object",
            "properties": {"text": {"type": "string"}},
            "required": ["text"],
        },
        "fn": lambda text: text[::-1],
    },
}

def handle(request: dict) -> dict:
    """Dispatch one JSON-RPC 2.0 request and return the response."""
    rid, method, params = request.get("id"), request["method"], request.get("params", {})

    if method == "initialize":
        result = {
            "protocolVersion": "2025-06-18",
            "capabilities": {"tools": {}},          # we only advertise tools
            "serverInfo": {"name": "toy-server", "version": "0.1.0"},
        }
    elif method == "tools/list":
        result = {"tools": [
            {"name": n, "description": t["description"], "inputSchema": t["inputSchema"]}
            for n, t in TOOLS.items()
        ]}
    elif method == "tools/call":
        name, args = params["name"], params.get("arguments", {})
        value = TOOLS[name]["fn"](**args)
        # Tool results are returned as a list of typed content blocks.
        result = {"content": [{"type": "text", "text": str(value)}], "isError": False}
    else:
        return {"jsonrpc": "2.0", "id": rid,
                "error": {"code": -32601, "message": f"Method not found: {method}"}}

    return {"jsonrpc": "2.0", "id": rid, "result": result}


# --- Simulate a client session ---------------------------------------------
session = [
    {"jsonrpc": "2.0", "id": 1, "method": "initialize",
     "params": {"protocolVersion": "2025-06-18", "capabilities": {},
                "clientInfo": {"name": "demo-client", "version": "1.0"}}},
    {"jsonrpc": "2.0", "id": 2, "method": "tools/list"},
    {"jsonrpc": "2.0", "id": 3, "method": "tools/call",
     "params": {"name": "add", "arguments": {"a": 2, "b": 40}}},
    {"jsonrpc": "2.0", "id": 4, "method": "tools/call",
     "params": {"name": "reverse", "arguments": {"text": "context"}}},
]

for req in session:
    resp = handle(req)
    print(f"--> {req['method']}")
    print(f"<-- {json.dumps(resp['result'], separators=(',', ':'))}\n")


### Example 2 — Bridging MCP tools to an LLM's tool-use API

A host fetches the server's `tools/list`, hands those schemas to the model, and when the model emits a tool call, the host turns it back into an MCP `tools/call`. Here we show that round-trip and the small schema translation MCP → Anthropic tool-use format (they're nearly identical, which is the point).


In [ ]:
# 1. Host calls tools/list on the server (reuse the server from Example 1).
listed = handle({"jsonrpc": "2.0", "id": 10, "method": "tools/list"})["result"]["tools"]

# 2. Translate MCP tool defs -> Anthropic Messages API tool format.
#    MCP uses `inputSchema`; Anthropic uses `input_schema`. Otherwise identical.
def to_anthropic(mcp_tool: dict) -> dict:
    return {
        "name": mcp_tool["name"],
        "description": mcp_tool["description"],
        "input_schema": mcp_tool["inputSchema"],
    }

anthropic_tools = [to_anthropic(t) for t in listed]
print("Tools passed to the model:")
print(json.dumps(anthropic_tools, indent=2))

# 3. Pretend the model decided to call a tool (this block is what an LLM emits).
model_tool_use = {"type": "tool_use", "id": "tu_01", "name": "add",
                  "input": {"a": 19, "b": 23}}

# 4. Host routes that decision back into an MCP tools/call and returns the result.
mcp_resp = handle({"jsonrpc": "2.0", "id": 11, "method": "tools/call",
                   "params": {"name": model_tool_use["name"],
                              "arguments": model_tool_use["input"]}})
tool_result_text = mcp_resp["result"]["content"][0]["text"]

# 5. Feed the result back to the model as a tool_result content block.
tool_result_block = {"type": "tool_result", "tool_use_id": model_tool_use["id"],
                     "content": tool_result_text}
print("\nModel asked for:", model_tool_use["name"], model_tool_use["input"])
print("tool_result returned to model:", json.dumps(tool_result_block))


### Example 3 — The real SDK (FastMCP)

This is the idiomatic way to author a server in Python. The decorator generates the `inputSchema` from your type hints automatically. The cell runs only if the `mcp` SDK is installed; otherwise it prints the shape so the notebook still executes top-to-bottom.


In [ ]:
import importlib.util

SNIPPET = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("toy-server")          # name shown to clients

@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers."""          # docstring becomes the tool description
    return a + b

@mcp.resource("greeting://{name}")
def greeting(name: str) -> str:
    """A read-only resource addressed by URI."""
    return f"Hello, {name}!"

if __name__ == "__main__":
    mcp.run()                        # serves over stdio by default
'''

if importlib.util.find_spec("mcp") is not None:
    # Build the server object and introspect the tool the decorator registered.
    import asyncio
    from mcp.server.fastmcp import FastMCP

    server = FastMCP("toy-server")

    @server.tool()
    def add(a: float, b: float) -> float:
        """Add two numbers."""
        return a + b

    tools = asyncio.run(server.list_tools())
    print("Registered tool:", tools[0].name, "->", tools[0].description)
    print("Auto-generated inputSchema:")
    print(json.dumps(tools[0].inputSchema, indent=2))
else:
    print("mcp SDK not installed — here is the canonical server you would write:")
    print(SNIPPET)


## 6. Gotchas & Pitfalls

- **The server never calls the LLM.** A common mental slip is putting model logic in the server. Servers expose capabilities; the host owns the model. If a server *needs* a completion, it uses the **sampling** primitive to ask the host.
- **stdio servers must keep stdout clean.** With the stdio transport, JSON-RPC frames travel over stdout. A stray `print()` (or a library logging to stdout) corrupts the stream. **Log to stderr**, not stdout.
- **Tool descriptions are prompt engineering.** The model picks tools purely from their `name`/`description`/schema. Vague descriptions → wrong or skipped tool calls. Write them like you're writing for the model, because you are.
- **Trust and prompt injection.** A server's tool descriptions and returned content enter the model's context. A malicious or compromised server can attempt **tool-description injection** or return poisoned data. Only connect servers you trust; sandbox and scope credentials per server.
- **Capability negotiation is mandatory.** Don't call `tools/call` before `initialize` completes, and don't assume a peer supports a feature it didn't advertise.
- **Transport confusion.** The old HTTP+SSE transport is deprecated in favor of **Streamable HTTP**. For local tools, stdio is almost always what you want.
- **Versioning.** The protocol is dated (`protocolVersion`, e.g. `2025-06-18`). Client and server negotiate a common version; mismatches should be handled, not assumed away.
- **Statefulness.** A session is a live connection with state (negotiated capabilities, subscriptions). Reconnects start a fresh `initialize` — don't cache a session across process restarts.


## 7. When to Use vs Alternatives

| Approach | Best for | Trade-offs vs MCP |
|---|---|---|
| **MCP** | Reusable tools/data shared across many hosts; clean process/trust boundary; growing plug-in ecosystem | Adds a process + protocol layer; overkill for one in-process tool |
| **Native function calling / tool use** (Anthropic, OpenAI) | A single app with a few tightly-coupled tools | No standard reuse across apps; you re-wrap tools per framework. MCP servers can *feed* these APIs (Example 2) |
| **Framework tools** (LangChain `Tool`, LlamaIndex, CrewAI) | Staying inside one framework's abstractions | Framework lock-in; most now also *consume* MCP servers, so MCP is complementary, not competing |
| **OpenAPI / plugins** | Exposing an existing REST API to a model | Designed for HTTP services, not local processes; no resources/prompts/sampling primitives; heavier spec |
| **A2A (Agent-to-Agent)** | Communication *between autonomous agents* | Different layer: A2A is agent↔agent; MCP is agent↔tool/data. They compose — see [[agent-to-agent]] |

**Rule of thumb:** if a capability should be usable by more than one client, or needs to run with its own permissions, make it an MCP server. If it's one function inside one app, just use the model's native tool-use API. MCP and function calling aren't rivals — MCP is how you *deliver* tools that the function-calling loop then *invokes*.

Related notebooks: [[langchain]], [[langgraph]], [[openai-agents-sdk]], [[agent-to-agent]].


## 8. Resources

- **Official spec & docs** — https://modelcontextprotocol.io/  (concepts, architecture, lifecycle)
- **Specification (dated, normative)** — https://modelcontextprotocol.io/specification/2025-06-18
- **Python SDK (FastMCP)** — https://github.com/modelcontextprotocol/python-sdk
- **Reference servers** (filesystem, git, fetch, …) — https://github.com/modelcontextprotocol/servers
- **Anthropic announcement** — https://www.anthropic.com/news/model-context-protocol
- **Inspector** (debug servers interactively) — https://github.com/modelcontextprotocol/inspector


YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
SUPPORTED = ("2025-06-18", "2025-03-26")


def serve(requests, tools):
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE